# Exercícios — Algoritmos de Ordenação — Parte II
## Merge Sort, Quick Sort e análise experimental com C + Python

Nesta segunda parte, continuaremos usando **C para implementar e medir os algoritmos** e **Python para organizar, filtrar, combinar e visualizar os resultados**.

### Objetivos

- implementar Merge Sort em C;
- implementar Quick Sort em C;
- compreender a estratégia de **divisão e conquista**;
- comparar entradas aleatórias, ordenadas e inversamente ordenadas;
- observar experimentalmente o efeito da escolha do pivô no Quick Sort;
- aprofundar o uso de `pandas`;
- combinar arquivos CSV com `pd.concat`;
- criar gráficos comparativos com `matplotlib`;
- relacionar os resultados com \(O(n \log n)\) e \(O(n^2)\).

## 1. Preparação

O arquivo de cabeçalho abaixo define a mesma estrutura de estatísticas usada na Parte I e acrescenta os protótipos do Merge Sort e Quick Sort.

Se você estiver executando este notebook de forma independente, rode esta célula antes das demais.

In [ ]:
%%writefile ordenacao.h
#ifndef ORDENACAO_H
#define ORDENACAO_H

typedef struct {
    long long comparacoes;
    long long trocas;
} Estatisticas;

void merge_sort(int v[], int n, Estatisticas *est);
void quick_sort(int v[], int n, Estatisticas *est);

#endif

# Exercício Prático 1 — Merge Sort

O Merge Sort utiliza a estratégia **dividir para conquistar**:

1. divide o vetor em duas partes;
2. ordena recursivamente cada metade;
3. combina (`merge`) as duas metades ordenadas.

Pseudocódigo simplificado:

```text
MergeSort(L, esquerda, direita)
    se esquerda < direita
        meio = (esquerda + direita) / 2

        MergeSort(L, esquerda, meio)
        MergeSort(L, meio+1, direita)

        Merge(L, esquerda, meio, direita)
```

Na operação `Merge`, compare os elementos das duas metades e copie para um vetor auxiliar na ordem correta.

### Sua tarefa

Implemente `merge_sort` no arquivo abaixo. Você pode criar funções auxiliares `static` para realizar a recursão e o `merge`.

Registre:

- comparações entre elementos em `est->comparacoes`;
- cópias/movimentações em `est->trocas`.

In [ ]:
%%writefile merge_impl.c
#include <stdlib.h>
#include "ordenacao.h"

/* Você pode criar funções auxiliares static aqui. */

void merge_sort(int v[], int n, Estatisticas *est) {
    /* TODO:
       implemente o Merge Sort.
    */
}

In [ ]:
%%writefile teste_merge.c
#include <stdio.h>
#include <stdlib.h>
#include "ordenacao.h"

int esta_ordenado(const int v[], int n) {
    for (int i = 1; i < n; i++)
        if (v[i - 1] > v[i])
            return 0;
    return 1;
}

int main(void) {
    int v[] = {38, 27, 43, 3, 9, 82, 10, 5, 1, 14};
    int n = sizeof(v) / sizeof(v[0]);

    Estatisticas est = {0, 0};

    merge_sort(v, n, &est);

    printf("Resultado: ");
    for (int i = 0; i < n; i++)
        printf("%d ", v[i]);

    printf("\nCorreto? %s\n", esta_ordenado(v, n) ? "SIM" : "NAO");
    printf("Comparacoes: %lld | Movimentacoes: %lld\n",
           est.comparacoes, est.trocas);

    return 0;
}

In [ ]:
!gcc -Wall -Wextra -O2 -std=c11 merge_impl.c teste_merge.c -o teste_merge
!./teste_merge

# Exercício Prático 2 — Quick Sort

Nesta atividade, manteremos a escolha do notebook original: **o primeiro elemento da região será usado como pivô**.

A partição será:

```text
Particao(L, p, r)
    valor = L[p]
    i = p

    para j = p+1 até r
        se L[j] <= valor
            i = i + 1
            troca L[i], L[j]

    troca L[i], L[p]
    retorna i
```

Após a partição, o pivô está definitivamente na posição `q`. Portanto, as chamadas recursivas usadas nesta adaptação são:

```text
QuickSort(L, p, q-1)
QuickSort(L, q+1, r)
```

> **Observação:** no pseudocódigo do notebook original, a primeira chamada aparece como `QuickSort(L,p,q)`. Nesta versão usamos `q-1`, pois `q` já contém o pivô em sua posição final.

### Sua tarefa

Implemente `quick_sort`. Registre as comparações e trocas realizadas.

In [ ]:
%%writefile quick_impl.c
#include "ordenacao.h"

/* Você pode criar funções auxiliares static aqui. */

void quick_sort(int v[], int n, Estatisticas *est) {
    /* TODO:
       implemente o Quick Sort usando o PRIMEIRO elemento como pivô.
    */
}

In [ ]:
%%writefile teste_quick.c
#include <stdio.h>
#include "ordenacao.h"

int esta_ordenado(const int v[], int n) {
    for (int i = 1; i < n; i++)
        if (v[i - 1] > v[i])
            return 0;
    return 1;
}

int main(void) {
    int v[] = {38, 27, 43, 3, 9, 82, 10, 5, 1, 14};
    int n = sizeof(v) / sizeof(v[0]);

    Estatisticas est = {0, 0};

    quick_sort(v, n, &est);

    printf("Resultado: ");
    for (int i = 0; i < n; i++)
        printf("%d ", v[i]);

    printf("\nCorreto? %s\n", esta_ordenado(v, n) ? "SIM" : "NAO");
    printf("Comparacoes: %lld | Trocas: %lld\n",
           est.comparacoes, est.trocas);

    return 0;
}

In [ ]:
!gcc -Wall -Wextra -O2 -std=c11 quick_impl.c teste_quick.c -o teste_quick
!./teste_quick

# Exercício Prático 3 — Experimentos: Merge Sort × Quick Sort

No notebook original, Merge Sort e Quick Sort são comparados para listas ordenadas e inversamente ordenadas. Nesta adaptação acrescentaremos também **entradas aleatórias**.

Isso é particularmente importante para o Quick Sort desta atividade: como escolhemos o **primeiro elemento como pivô**, vetores já ordenados ou inversamente ordenados podem produzir partições muito desequilibradas.

O programa C abaixo executará os dois algoritmos para diferentes tamanhos, repetirá cada experimento e gravará os resultados em:

```text
resultados_parte2.csv
```

> Só execute o benchmark depois de obter `Correto? SIM` nos dois exercícios anteriores.

In [ ]:
%%writefile benchmark_parte2.c
#include <stdio.h>
#include <stdlib.h>
#include <string.h>
#include <time.h>
#include "ordenacao.h"

#define REPETICOES 5

typedef void (*Algoritmo)(int [], int, Estatisticas *);

int esta_ordenado(const int v[], int n) {
    for (int i = 1; i < n; i++)
        if (v[i - 1] > v[i])
            return 0;
    return 1;
}

void gera_entrada(int v[], int n, const char *tipo) {
    if (strcmp(tipo, "ordenado") == 0) {
        for (int i = 0; i < n; i++)
            v[i] = i;
    } else if (strcmp(tipo, "inverso") == 0) {
        for (int i = 0; i < n; i++)
            v[i] = n - i;
    } else {
        for (int i = 0; i < n; i++)
            v[i] = rand();
    }
}

double mede_tempo_us(Algoritmo algoritmo, int v[], int n,
                     Estatisticas *est) {
    clock_t inicio = clock();
    algoritmo(v, n, est);
    clock_t fim = clock();

    return 1e6 * (double)(fim - inicio) / CLOCKS_PER_SEC;
}

int main(void) {
    const int tamanhos[] = {500, 1000, 2000, 4000, 8000};
    const int nt = sizeof(tamanhos) / sizeof(tamanhos[0]);

    const char *tipos[] = {"aleatorio", "ordenado", "inverso"};

    const char *nomes[] = {"merge", "quick"};
    Algoritmo algoritmos[] = {merge_sort, quick_sort};

    FILE *arq = fopen("resultados_parte2.csv", "w");
    if (!arq) {
        perror("Nao foi possivel criar o CSV");
        return 1;
    }

    fprintf(arq,
            "algoritmo,n,tipo,execucao,tempo_us,comparacoes,trocas\n");

    srand(42);

    for (int a = 0; a < 2; a++) {
        for (int t = 0; t < 3; t++) {
            for (int s = 0; s < nt; s++) {
                int n = tamanhos[s];

                int *base = malloc(n * sizeof(int));
                int *v = malloc(n * sizeof(int));

                gera_entrada(base, n, tipos[t]);

                for (int r = 1; r <= REPETICOES; r++) {
                    memcpy(v, base, n * sizeof(int));

                    Estatisticas est = {0, 0};
                    double tempo = mede_tempo_us(
                        algoritmos[a], v, n, &est);

                    if (!esta_ordenado(v, n)) {
                        fprintf(stderr,
                                "ERRO: %s nao ordenou corretamente "
                                "(n=%d, tipo=%s).\n",
                                nomes[a], n, tipos[t]);
                        fclose(arq);
                        free(base);
                        free(v);
                        return 2;
                    }

                    fprintf(arq, "%s,%d,%s,%d,%.3f,%lld,%lld\n",
                            nomes[a], n, tipos[t], r, tempo,
                            est.comparacoes, est.trocas);
                }

                free(base);
                free(v);
            }
        }
    }

    fclose(arq);
    printf("Resultados gravados em resultados_parte2.csv\n");
    return 0;
}

In [ ]:
!gcc -Wall -Wextra -O2 -std=c11 \
    merge_impl.c quick_impl.c benchmark_parte2.c \
    -o benchmark_parte2

!./benchmark_parte2

# 4. Python: lendo e inspecionando o arquivo de resultados

Vamos novamente utilizar `pandas`.

```python
df2 = pd.read_csv("resultados_parte2.csv")
```

O resultado é um `DataFrame`.

Além de `head()`, duas operações úteis para começar a investigar um conjunto de dados são:

- `df.shape` — informa `(número de linhas, número de colunas)`;
- `df["coluna"].unique()` — mostra quais valores diferentes aparecem em uma coluna.

Esses comandos ajudam a entender o arquivo antes de realizar a análise.

In [ ]:
import pandas as pd

df2 = pd.read_csv("resultados_parte2.csv")

display(df2.head())

print("Dimensões:", df2.shape)
print("Algoritmos:", df2["algoritmo"].unique())
print("Tipos de entrada:", df2["tipo"].unique())
print("Tamanhos:", df2["n"].unique())

## 4.1 Filtragem com mais de uma condição

Em `pandas`, podemos combinar condições usando:

- `&` para **E**;
- `|` para **OU**;
- `~` para **NÃO**.

Cada condição deve ficar entre parênteses.

Exemplo:

```python
df2[
    (df2["algoritmo"] == "quick") &
    (df2["tipo"] == "ordenado")
]
```

A célula seguinte seleciona apenas execuções do Quick Sort sobre entradas ordenadas.

In [ ]:
quick_ordenado = df2[
    (df2["algoritmo"] == "quick") &
    (df2["tipo"] == "ordenado")
]

display(quick_ordenado.head(10))

## 4.2 Resumindo as repetições

Assim como na Parte I, usaremos `groupby`.

Observe a leitura da expressão abaixo de dentro para fora:

1. agrupar por `algoritmo`, `n` e `tipo`;
2. calcular médias;
3. usar `reset_index()` para obter novamente uma tabela comum.

Essa é uma operação muito frequente em análise de dados com Python.

In [ ]:
resumo2 = (
    df2.groupby(["algoritmo", "n", "tipo"])
       .agg(
           tempo_medio_us=("tempo_us", "mean"),
           comparacoes_medias=("comparacoes", "mean"),
           trocas_medias=("trocas", "mean"),
       )
       .reset_index()
)

display(resumo2.head(15))

# 5. Python: gráficos comparando Merge Sort e Quick Sort

O primeiro gráfico considera **entradas aleatórias**.

A estratégia utilizada é:

1. filtrar o `DataFrame`;
2. obter os nomes diferentes de algoritmos com `.unique()`;
3. percorrer os algoritmos com `for`;
4. desenhar uma curva para cada um com `plt.plot()`.

Esse padrão será reutilizado várias vezes.

In [ ]:
import matplotlib.pyplot as plt

dados = resumo2[resumo2["tipo"] == "aleatorio"]

plt.figure(figsize=(8, 5))

for algoritmo in dados["algoritmo"].unique():
    parte = dados[dados["algoritmo"] == algoritmo]
    plt.plot(
        parte["n"],
        parte["tempo_medio_us"],
        marker="o",
        label=algoritmo
    )

plt.xlabel("Tamanho do vetor (n)")
plt.ylabel("Tempo médio (microssegundos)")
plt.title("Merge Sort × Quick Sort — entradas aleatórias")
plt.legend()
plt.grid()
plt.show()

## 5.1 Efeito da organização inicial no Quick Sort

Agora filtraremos apenas `quick` e desenharemos uma curva para cada tipo de entrada.

Esse gráfico deve ser interpretado juntamente com a escolha do pivô. Pergunte-se:

- as partições produzidas pelo pivô são equilibradas?
- qual é a profundidade aproximada da recursão?
- o comportamento está próximo de \(O(n \log n)\) ou de \(O(n^2)\)?

In [ ]:
dados_quick = resumo2[resumo2["algoritmo"] == "quick"]

plt.figure(figsize=(8, 5))

for tipo in dados_quick["tipo"].unique():
    parte = dados_quick[dados_quick["tipo"] == tipo]
    plt.plot(
        parte["n"],
        parte["tempo_medio_us"],
        marker="o",
        label=tipo
    )

plt.xlabel("Tamanho do vetor (n)")
plt.ylabel("Tempo médio (microssegundos)")
plt.title("Quick Sort — efeito da organização da entrada")
plt.legend()
plt.grid()
plt.show()

## 5.2 `pivot_table`: reorganizando dados

`pandas` também permite reorganizar uma tabela para facilitar comparações.

Uma **tabela dinâmica** (`pivot_table`) pode transformar valores de uma coluna em novas colunas.

Por exemplo:

```python
resumo2.pivot_table(
    index="n",
    columns=["algoritmo", "tipo"],
    values="tempo_medio_us"
)
```

Aqui:

- `index="n"` faz com que cada linha corresponda a um tamanho;
- `columns=[...]` cria colunas para combinações de algoritmo e tipo;
- `values="tempo_medio_us"` define o valor armazenado nas células.

Esse formato é útil para inspeção e também para exportar tabelas de resultados.

In [ ]:
tabela_tempos = resumo2.pivot_table(
    index="n",
    columns=["algoritmo", "tipo"],
    values="tempo_medio_us"
)

display(tabela_tempos)

# 6. Python: combinando os resultados das Partes I e II

Se `resultados_parte1.csv` estiver disponível no ambiente, podemos combinar os cinco algoritmos.

A função:

```python
pd.concat([df1, df2], ignore_index=True)
```

concatena tabelas que possuem colunas compatíveis.

`ignore_index=True` pede ao `pandas` para criar uma nova numeração contínua para as linhas.

Essa operação é muito usada quando dados vêm de arquivos diferentes, mas possuem a mesma estrutura.

In [ ]:
from pathlib import Path

arquivo_parte1 = Path("resultados_parte1.csv")

if arquivo_parte1.exists():
    df1 = pd.read_csv(arquivo_parte1)

    todos = pd.concat(
        [df1, df2],
        ignore_index=True
    )

    print("Arquivos combinados com sucesso.")
    print("Algoritmos disponíveis:", todos["algoritmo"].unique())
    display(todos.head())
else:
    print(
        "resultados_parte1.csv não foi encontrado. "
        "Execute antes o notebook da Parte I para realizar "
        "a comparação entre os cinco algoritmos."
    )

## 6.1 Comparação geral dos algoritmos

Se os dois arquivos estiverem disponíveis, construa uma comparação geral.

Há um detalhe experimental importante: os tamanhos usados nas Partes I e II não são exatamente os mesmos para todos os algoritmos. Isso é intencional, pois algoritmos \(O(n^2)\) tornam-se caros mais rapidamente.

Uma forma de comparar é selecionar apenas tamanhos que aparecem nos dois conjuntos ou executar novamente os benchmarks usando uma lista de tamanhos comum.

In [ ]:
if arquivo_parte1.exists():
    resumo_todos = (
        todos.groupby(["algoritmo", "n", "tipo"])
             .agg(tempo_medio_us=("tempo_us", "mean"))
             .reset_index()
    )

    dados = resumo_todos[resumo_todos["tipo"] == "aleatorio"]

    plt.figure(figsize=(9, 5))

    for algoritmo in dados["algoritmo"].unique():
        parte = dados[dados["algoritmo"] == algoritmo]
        plt.plot(
            parte["n"],
            parte["tempo_medio_us"],
            marker="o",
            label=algoritmo
        )

    plt.xlabel("Tamanho do vetor (n)")
    plt.ylabel("Tempo médio (microssegundos)")
    plt.title("Comparação geral — entradas aleatórias")
    plt.legend()
    plt.grid()
    plt.show()

# Exercício Prático 4 — Melhorando o Quick Sort

O Quick Sort atual sempre utiliza o **primeiro elemento como pivô**.

Implemente uma segunda versão utilizando uma das estratégias:

- pivô aleatório;
- elemento central;
- mediana de três.

Depois:

1. execute novamente os experimentos;
2. acrescente a nova versão ao CSV com outro nome, por exemplo `quick_random`;
3. compare as versões para entradas aleatórias, ordenadas e inversas;
4. explique as diferenças observadas.

Essa atividade mostra que a complexidade de um algoritmo não depende apenas de seu nome: **decisões de implementação e características da entrada também importam**.

# 7. Questões finais de análise

1. Compare Merge Sort e Quick Sort para entradas aleatórias.
2. O Merge Sort sofre grande influência da organização inicial da entrada?
3. O que ocorre com o Quick Sort de pivô inicial quando a lista já está ordenada?
4. O mesmo comportamento aparece para listas em ordem inversa?
5. Compare os gráficos de **tempo** e de **número de comparações**.
6. Explique experimentalmente a diferença entre \(O(n^2)\) e \(O(n\log n)\).
7. Em quais situações Insertion Sort pode parecer competitivo mesmo tendo complexidade \(O(n^2)\) no pior caso?
8. Selection Sort faz menos comparações quando a entrada já está ordenada? Justifique usando os dados da Parte I.
9. O que mudou quando você alterou a estratégia de escolha do pivô?
10. Quais limitações existem em usar apenas tempo de execução como evidência experimental?

## Desafio opcional — escala logarítmica

Quando os tempos variam muito, uma escala logarítmica pode facilitar a visualização.

No `matplotlib`, podemos usar:

```python
plt.yscale("log")
```

para tornar o eixo vertical logarítmico, ou:

```python
plt.xscale("log")
plt.yscale("log")
```

para usar escala logarítmica nos dois eixos.

Experimente repetir o gráfico geral usando escala logarítmica. Compare a legibilidade com o gráfico original e explique quando essa transformação é útil.